# 3. Graphiques interactifs avec Plotly

Plotly.js est utilisé par défaut. On génère du HTML contenant le graphique et on l'affiche avec `Deno.jupyter.display`.

## 3.0 Fonction d'affichage

La fonction `plotly(data, layout, nom)` ci-dessous affiche un graphique [Plotly.js](https://plotly.com/javascript/), avec deux boutons **⬇ PNG** et **⬇ SVG** pour le télécharger (`nom` est le nom du fichier). Exécute cette cellule une fois, puis réutilise la fonction dans les cellules suivantes.

In [ ]:
// data : séries du graphique ; layout : titre, axes...
// nom : nom du fichier téléchargé avec les boutons PNG et SVG.
function plotly(data: object[], layout: object = {}, nom = "graphique") {
  const id = `plot-${crypto.randomUUID()}`;
  const html = `
<div id="${id}" style="width: 100%; max-width: 750px; height: 450px;"></div>
<div style="margin: 4px 0 16px;">
  <button id="${id}-png">⬇ PNG</button>
  <button id="${id}-svg">⬇ SVG</button>
</div>
<script>
  (function () {
    const id = ${JSON.stringify(id)};
    const data = ${JSON.stringify(data)};
    const layout = ${JSON.stringify(layout)};
    const nom = ${JSON.stringify(nom)};

    function draw() {
      Plotly.newPlot(id, data, layout, {
        responsive: true,
        toImageButtonOptions: { filename: nom, scale: 2 },
      });
      for (const format of ["png", "svg"]) {
        const button = document.getElementById(id + "-" + format);
        button.onclick = () =>
          Plotly.downloadImage(id, { format, filename: nom, scale: 2 });
      }
    }

    if (window.Plotly) return draw();
    const script = document.createElement("script");
    script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
    script.onload = draw;
    document.head.appendChild(script);
  })();
</script>`;
  Deno.jupyter.display({ "text/html": html }, { raw: true });
}

## 3.1 Graphique en barres

In [ ]:
plotly(
  [{
    type: "bar",
    x: ["Littéraire", "Scientifique", "Économique", "Artistique"],
    y: [45, 80, 60, 30],
    marker: { color: ["#ff6384", "#36a2eb", "#ffce56", "#4bc0c0"] },
  }],
  {
    title: "Répartition des élèves par filière",
    xaxis: { title: "Filières" },
    yaxis: { title: "Nombre d'élèves" },
  },
  "eleves-par-filiere",
);

## 3.2 Graphique en secteurs (camembert)

In [ ]:
plotly(
  [{
    type: "pie",
    labels: ["Apple", "Samsung", "Xiaomi", "Autres"],
    values: [40, 35, 15, 10],
  }],
  { title: "Parts de marché des fabricants de téléphones" },
  "parts-de-marche",
);

## 3.3 Nuage de points

In [ ]:
plotly(
  [{
    type: "scatter",
    mode: "markers",
    x: [1, 2, 3, 4, 5, 6, 7, 8],
    y: [40, 50, 55, 60, 70, 75, 80, 85],
    marker: { size: 12, color: "#36a2eb" },
  }],
  {
    title: "Corrélation entre heures d'étude et notes",
    xaxis: { title: "Heures d'étude" },
    yaxis: { title: "Note (%)" },
  },
  "heures-etude-notes",
);

## 3.4 Graphique de séries temporelles

In [ ]:
plotly(
  [{
    type: "scatter",
    mode: "lines+markers",
    x: ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"],
    y: [12, 14, 15, 13, 11, 9, 10],
    line: { color: "#4bc0c0" },
  }],
  {
    title: "Évolution des températures sur une semaine",
    xaxis: { title: "Jour" },
    yaxis: { title: "Température (°C)" },
  },
  "temperatures-semaine",
);

## 3.5 Histogramme

In [ ]:
plotly(
  [{
    type: "histogram",
    x: [15, 17, 18, 19, 21, 21, 22, 23, 24, 24, 25, 25, 26, 27, 30],
    nbinsx: 6,
    marker: { color: "#9966ff" },
  }],
  {
    title: "Distribution des âges des participants",
    xaxis: { title: "Âge" },
    yaxis: { title: "Fréquence" },
  },
  "distribution-ages",
);

## 3.6 Graphique avec des données réelles (Polars + Plotly)

In [ ]:
import pl from "nodejs-polars";

const df = pl.readCSV("../data/titanic.csv");

const survivalByClass = df
  .groupBy("Pclass")
  .agg(pl.col("Survived").mean().alias("survival_rate"))
  .sort("Pclass");

const rows = survivalByClass.toRecords();
const classes = rows.map((r) => `Classe ${r.Pclass}`);
const rates = rows.map((r) => Number(r.survival_rate));

const data = [{
  type: "bar",
  x: classes,
  y: rates,
  marker: { color: ["#ff6384", "#36a2eb", "#ffce56"] },
}];

const layout = {
  title: "Taux de survie par classe (Titanic)",
  xaxis: { title: "Classe" },
  yaxis: { title: "Taux de survie", range: [0, 1] },
};

plotly(data, layout, "titanic-survie-par-classe");

## 3.7 Télécharger un graphique

Sous chaque graphique, deux boutons enregistrent l'image dans le dossier *Téléchargements* :

- **⬇ PNG** : image classique, à coller dans un document ;
- **⬇ SVG** : image vectorielle, nette à toutes les tailles (idéale pour une présentation).

Le fichier porte le nom donné en 3e argument de `plotly` (par exemple `"titanic-survie-par-classe"`). L'icône 📷 de la barre d'outils Plotly, visible en survolant le graphique, télécharge aussi le graphique en PNG.